# Step 1 : Groupby - Average

In [1]:
# Necessary Libraries
import pandas as pd
import os
from datetime import datetime, time, timedelta
import math
import numpy as np

## Data Awal

### Data Reel

In [2]:
# Pipeline for Data Reel
def merge_reel_data(file_list):
    dataframes = []
    
    for file in file_list:
        if os.path.exists(file):
            df = pd.read_excel(file)
            dataframes.append(df)
            print(f"Berhasil memuat: {file} | Shape: {df.shape}")
        else:
            print(f"GAGAL MEMUAT: {file} tidak ditemukan di direktori.")
            
    if not dataframes:
        raise ValueError("Pipeline dihentikan. Tidak ada satupun file yang valid untuk digabungkan.")
        
    master_reel = pd.concat(dataframes, ignore_index=True)
    
    return master_reel

In [3]:
# Eksekusi Pipeline
daftar_file_reel = [
    "../Data Reel/data reel pm15 0326.xlsx",
    "../Data Reel/data reel pm15 0426.xlsx"
]

reel_pm15 = merge_reel_data(daftar_file_reel)
reel_pm15.head()

Berhasil memuat: ../Data Reel/data reel pm15 0326.xlsx | Shape: (698, 14)
Berhasil memuat: ../Data Reel/data reel pm15 0426.xlsx | Shape: (696, 14)


,Time,Tanggal,Grade,Shift,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,7.30,01.03.26,T 14.2.Recycle,1,98,14.18,0.89,595,403,83,0.139496,27,81.3,Acc Sotiss
1,8.32,01.03.26,T 14.2.Recycle,1,99,14.78,0.91,788,489,105,0.133249,27,81.3,Acc Sotiss
2,9.40,01.03.26,T 14.2.Recycle,1,1,14.60,0.94,874,427,106,0.121281,30,81.0,Pending
3,10.18,01.03.26,T 14.2.Recycle,1,2,14.84,0.90,865,394,115,0.132948,31,81.8,Pending
4,13.17,01.03.26,T 14.2.Recycle,1,3,15.80,1.03,523,318,75,0.143403,33,81.7,Pending


### Params PM

In [4]:
# Pipeline for Params PM
def load_and_standardize(file_path):
    # Deteksi ekstensi file untuk metode ekstraksi yang tepat
    _, ext = os.path.splitext(file_path)
    
    if ext.lower() in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    elif ext.lower() == '.csv':
        with open(file_path, encoding='utf-16') as f:
            raw_header = f.readline().strip()
            # Header bungkus kutip ganda — bersihkan manual
            header = raw_header.strip('"').replace('""', '').split(';')
            df = pd.read_csv(f, delimiter=';', names=header)
    else:
        raise ValueError(f"Format file tidak dikenali: {ext}")

    # Standarisasi Header
    df.columns = df.columns.str.strip()
    
    # ---------------------------------------------------------
    if ext.lower() in ['.xlsx', '.xls']:
        df.rename(columns={'Load KWH Refiner': 'Load KWH Tickling Refiner'}, inplace=True)
    # ---------------------------------------------------------

    rename_map = {
        'Yangkee  Speed': 'Yankee Speed',
        'Coating Flow': 'Flow Coating',
        'Release Flow': 'Flow Release',
        'Jet Rasio': 'Jet Wire Ratio',
        'Yangkee Temperatur' : 'Yankee Temperature',
        'Hood Tempetatur' : 'Hood Temperature',
        'Load Ampere Turbo Vakum' : 'Load Ampere Turbo Vacuum'
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
    
    # Validasi Waktu sebagai Kunci Utama
    df.dropna(subset=['Time'], inplace=True) 
    df['Time'] = pd.to_datetime(df['Time'], dayfirst=True, format='mixed', errors='coerce')
    
    return df

def build_master_pipeline(file_list):
    dataframes = []
    
    for file in file_list:
        try:
            df = load_and_standardize(file)
            dataframes.append(df)
        except FileNotFoundError:
            print(f"Peringatan: File {file} tidak ditemukan. Dilewati.")
            
    if not dataframes:
        raise ValueError("Tidak ada data yang berhasil dimuat.")

    # Penggabungan dan Pembersihan Data Redundan
    master_df = pd.concat(dataframes, ignore_index=True)
    master_df.drop_duplicates(subset=['Time'], keep='last', inplace=True)
    master_df.sort_values('Time', inplace=True)
    
    return master_df.reset_index(drop=True)



In [5]:
# Eksekusi Pipeline
file_sources = [
    '../PM Params/Maret-April PM 15.xlsx',
    '../PM Params/04052026_PM15.csv'
]
raw15 = build_master_pipeline(file_sources)
raw15.drop(['Yankee Temperature', 'Load KWH Refiner', 'Hood Temperature', 'Load Ampere Turbo Vacuum'], axis=1, inplace=True)
raw15.info()

<class 'pandas.DataFrame'>
RangeIndex: 86345 entries, 0 to 86344
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Time                       86345 non-null  datetime64[us]
 1   Yankee Speed               86345 non-null  float64       
 2   Pope Reel Speed            86345 non-null  float64       
 3   Yankee Pressure            86345 non-null  float64       
 4   Stock Flow                 86345 non-null  float64       
 5   Stock Consistency          86345 non-null  float64       
 6   Flow Coating               86345 non-null  float64       
 7   Flow Release               86345 non-null  float64       
 8   Jet Wire Ratio             86345 non-null  float64       
 9   Load KWH Tickling Refiner  86345 non-null  float64       
dtypes: datetime64[us](1), float64(9)
memory usage: 6.6 MB


In [6]:
# Create Categories PM Stop/Run
raw15['PM_stop'] = np.where(raw15['Pope Reel Speed'] == 0, "stop", "run")

# Create Coating/(Area.Min) Feature
raw15['Coating/(Area.Min)'] = ((raw15['Flow Coating'] * 60) / (raw15['Yankee Speed'] * 3050))

In [7]:
raw15.info()

<class 'pandas.DataFrame'>
RangeIndex: 86345 entries, 0 to 86344
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Time                       86345 non-null  datetime64[us]
 1   Yankee Speed               86345 non-null  float64       
 2   Pope Reel Speed            86345 non-null  float64       
 3   Yankee Pressure            86345 non-null  float64       
 4   Stock Flow                 86345 non-null  float64       
 5   Stock Consistency          86345 non-null  float64       
 6   Flow Coating               86345 non-null  float64       
 7   Flow Release               86345 non-null  float64       
 8   Jet Wire Ratio             86345 non-null  float64       
 9   Load KWH Tickling Refiner  86345 non-null  float64       
 10  PM_stop                    86345 non-null  str           
 11  Coating/(Area.Min)         84708 non-null  float64       
dtypes: datetime64[u

#### Filter Out

In [8]:
# Filter Out Data
raw15 = raw15[(raw15['Yankee Speed'] >= 500) & (raw15['Pope Reel Speed'] >= 400)]

In [9]:
raw15.info()

<class 'pandas.DataFrame'>
Index: 80147 entries, 153 to 86344
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Time                       80147 non-null  datetime64[us]
 1   Yankee Speed               80147 non-null  float64       
 2   Pope Reel Speed            80147 non-null  float64       
 3   Yankee Pressure            80147 non-null  float64       
 4   Stock Flow                 80147 non-null  float64       
 5   Stock Consistency          80147 non-null  float64       
 6   Flow Coating               80147 non-null  float64       
 7   Flow Release               80147 non-null  float64       
 8   Jet Wire Ratio             80147 non-null  float64       
 9   Load KWH Tickling Refiner  80147 non-null  float64       
 10  PM_stop                    80147 non-null  str           
 11  Coating/(Area.Min)         80147 non-null  float64       
dtypes: datetime64[us](

In [10]:
# Delete before-after 0 values in 'Pope Reel Speed'
raw15 = raw15.reset_index(drop=True)
is_zero = np.isclose(raw15['Pope Reel Speed'], 0, atol=1e-5)
mask_to_drop = pd.Series(is_zero).rolling(window=11, center=True, min_periods=1).max().astype(bool)
df_clean = raw15[~mask_to_drop].copy()

In [11]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 80147 entries, 0 to 80146
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Time                       80147 non-null  datetime64[us]
 1   Yankee Speed               80147 non-null  float64       
 2   Pope Reel Speed            80147 non-null  float64       
 3   Yankee Pressure            80147 non-null  float64       
 4   Stock Flow                 80147 non-null  float64       
 5   Stock Consistency          80147 non-null  float64       
 6   Flow Coating               80147 non-null  float64       
 7   Flow Release               80147 non-null  float64       
 8   Jet Wire Ratio             80147 non-null  float64       
 9   Load KWH Tickling Refiner  80147 non-null  float64       
 10  PM_stop                    80147 non-null  str           
 11  Coating/(Area.Min)         80147 non-null  float64       
dtypes: datetime64[u

In [12]:
df_clean.head()

,Time,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Tickling Refiner,PM_stop,Coating/(Area.Min)
0,2026-01-04 02:33:00,949.828,576.037,7.673,863.087,2.999,21.068,37.974,0.888,251.415,run,0.000436
1,2026-01-04 02:34:00,950.144,818.912,7.864,913.747,3.015,21.071,37.978,0.888,250.731,run,0.000436
2,2026-01-04 02:35:00,949.934,816.942,8.082,952.638,3.055,21.066,37.970,0.888,252.284,run,0.000436
3,2026-01-04 02:36:00,949.934,817.150,8.034,988.711,3.099,21.066,37.970,0.888,251.468,run,0.000436
4,2026-01-04 02:37:00,949.828,816.942,7.980,1019.279,3.129,21.071,37.978,0.888,250.349,run,0.000436


## Preparation

### Preparation Process

#### Parameter PM

In [13]:
shift1_start = time(7, 0, 1)
shift1_end   = time(15, 0, 0)
shift2_start = time(15, 0, 1)
shift2_end   = time(23, 0, 0)
def assign_shift(t):
    if shift1_start <= t <= shift1_end:
        return 'Shift 1'
    elif shift2_start <= t <= shift2_end:
        return 'Shift 2'
    else:
        return 'Shift 3'

In [14]:
urutan_params_pm = [
    'Date', 'Time', 'Shift', 'Join_Key', 'Timestamp',
    'Creping', 'Yankee Speed', 'Pope Reel Speed',
    'Yankee Pressure', 'Stock Flow', 'Stock Consistency',
    'Flow Coating', 'Flow Release', 'Jet Wire Ratio',
    'Load KWH Tickling Refiner','Key_Date', 'PM_stop', 'Coating/(Area.Min)'
]

In [15]:
def preprocess_pm(df_input):
    df = df_input.copy()
    df = df.drop(columns=['Time'])
    df.columns = df.columns.str.strip()
    df.insert(0, 'Creping', (df['Yankee Speed'] - df['Pope Reel Speed']) * 100 / df['Yankee Speed'])
    def extract_time(x):
        if isinstance(x, str):
            return datetime.strptime(x.split(' ')[1], '%H:%M:%S').time()
        else:  # sudah datetime/Timestamp
            return x.time()
    df.insert(0, 'Time', df_input['Time'].apply(extract_time))
    df['Shift'] = df['Time'].apply(assign_shift)
    
    def extract_date(x):
        if isinstance(x, str):
            return x.split(' ')[0]
        else:
            return x.strftime('%d/%m/%y')
    df.insert(0, 'Date', df_input['Time'].apply(extract_date))
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%y')
    df['Key_Date'] = [df['Date'][index] - timedelta(days=1) 
                if
                df['Shift'][index] == 'Shift 3' 
                else df['Date'][index] 
                for index in range(len(df['Shift']))]
    df['Join_Key'] = df['Key_Date'].astype(str) + ' ' + df['Time'].astype(str) + ' ' + df['Shift']
    df['Timestamp'] = df['Date'].astype(str) + ' ' + df['Time'].astype(str)
    df['Shift'] = df['Shift'].str.extract(r'(\d+)').astype(int)
    df = df[urutan_params_pm]
    df.drop(columns=['Key_Date'], inplace=True)
    return df

#### Data Reel

In [16]:
def time_to_hms(val):
    s = str(val).strip()
    if s.lower() in {'', 'nan', 'none'}:
        return pd.NA

    # If already contains colon, parse parts directly
    if ':' in s:
        parts = s.split(':')
        h = int(parts[0])
        m = int(parts[1]) if len(parts) > 1 and parts[1] != '' else 0
        sec = int(parts[2]) if len(parts) > 2 and parts[2] != '' else 0

    # If contains dot, treat left as hours and right as minutes (common human shorthand)
    elif '.' in s:
        left, right = s.split('.', 1)
        if int(left) >= 24:
            return pd.NA  # Invalid hour value
        left = 0 if left == "24" else left  # Handle "24" as "00"
        h = int(left) if left != '' else 0

        # If right part is short (1 or 2 digits) treat it as minutes (e.g., "4.1" -> 4:01, "11.55" -> 11:55)
        if len(right) <= 2:
            m = int(right)
            sec = 0
        else:
            # If right part is longer, treat the whole value as a decimal hour (fallback)
            # e.g., "4.125" -> 4.125 hours -> convert fractional hour to minutes
            f = float(s)
            total_minutes = int(round((f - math.floor(f)) * 60))
            m = total_minutes
            sec = 0

    # No separator: treat as hours only (e.g., "6" -> 06:00:00)
    else:
        h = int(float(s))
        m = 0
        sec = 0

    # Normalize minutes >= 60 into hours
    if m >= 60:
        extra_h = m // 60
        h = (h + extra_h) % 24
        m = m % 60

    return f"{h:02d}:{m:02d}:{sec:02d}"

In [17]:
def preprocess_reel(df_input):
    df = df_input.copy()
    df['Time'] = df['Time'].apply(time_to_hms)
    df['Tanggal'] = pd.to_datetime(df['Tanggal'], format='%d.%m.%y')
    df = df.dropna(subset = ['Time']).reset_index(drop = True)
    df['Timestamp'] = df['Tanggal'].astype(str) + ' ' + df['Time'].astype(str)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S')
    # Edit Kolom
    df = df.rename(columns={'Tanggal': 'Date'})
    first_cols = ['Date', 'Time', 'Shift', 'Timestamp']
    other_cols = [col for col in df.columns if col not in first_cols]
    df = df[first_cols + other_cols]
    return df

### Apply Preparation

#### Parameter PM

In [18]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 80147 entries, 0 to 80146
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Time                       80147 non-null  datetime64[us]
 1   Yankee Speed               80147 non-null  float64       
 2   Pope Reel Speed            80147 non-null  float64       
 3   Yankee Pressure            80147 non-null  float64       
 4   Stock Flow                 80147 non-null  float64       
 5   Stock Consistency          80147 non-null  float64       
 6   Flow Coating               80147 non-null  float64       
 7   Flow Release               80147 non-null  float64       
 8   Jet Wire Ratio             80147 non-null  float64       
 9   Load KWH Tickling Refiner  80147 non-null  float64       
 10  PM_stop                    80147 non-null  str           
 11  Coating/(Area.Min)         80147 non-null  float64       
dtypes: datetime64[u

In [19]:
pm_15 = preprocess_pm(df_clean)
pm_15.head()

,Date,Time,Shift,Join_Key,Timestamp,Creping,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Tickling Refiner,PM_stop,Coating/(Area.Min)
0,2026-01-04,02:33:00,3,2026-01-03 02:33:00 Shift 3,2026-01-04 02:33:00,39.353546,949.828,576.037,7.673,863.087,2.999,21.068,37.974,0.888,251.415,run,0.000436
1,2026-01-04,02:34:00,3,2026-01-03 02:34:00 Shift 3,2026-01-04 02:34:00,13.811801,950.144,818.912,7.864,913.747,3.015,21.071,37.978,0.888,250.731,run,0.000436
2,2026-01-04,02:35:00,3,2026-01-03 02:35:00 Shift 3,2026-01-04 02:35:00,14.000131,949.934,816.942,8.082,952.638,3.055,21.066,37.970,0.888,252.284,run,0.000436
3,2026-01-04,02:36:00,3,2026-01-03 02:36:00 Shift 3,2026-01-04 02:36:00,13.978234,949.934,817.150,8.034,988.711,3.099,21.066,37.970,0.888,251.468,run,0.000436
4,2026-01-04,02:37:00,3,2026-01-03 02:37:00 Shift 3,2026-01-04 02:37:00,13.990533,949.828,816.942,7.980,1019.279,3.129,21.071,37.978,0.888,250.349,run,0.000436


#### Data Reel

In [20]:
reel_pm15 = preprocess_reel(reel_pm15)
reel_pm15.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,2026-03-01,07:03:00,1,2026-03-01 07:03:00,T 14.2.Recycle,98,14.18,0.89,595,403,83,0.139496,27,81.3,Acc Sotiss
1,2026-03-01,08:32:00,1,2026-03-01 08:32:00,T 14.2.Recycle,99,14.78,0.91,788,489,105,0.133249,27,81.3,Acc Sotiss
2,2026-03-01,09:04:00,1,2026-03-01 09:04:00,T 14.2.Recycle,1,14.60,0.94,874,427,106,0.121281,30,81.0,Pending
3,2026-03-01,10:18:00,1,2026-03-01 10:18:00,T 14.2.Recycle,2,14.84,0.90,865,394,115,0.132948,31,81.8,Pending
4,2026-03-01,13:17:00,1,2026-03-01 13:17:00,T 14.2.Recycle,3,15.80,1.03,523,318,75,0.143403,33,81.7,Pending


## Pipeline

In [21]:
# 1. MEMBACA DATA
df_reel = reel_pm15.copy()
df_params = pm_15.copy()

# 2. KONVERSI TIMESTAMP
df_reel['Timestamp'] = pd.to_datetime(df_reel['Timestamp'])
df_params['Timestamp'] = pd.to_datetime(df_params['Timestamp'])

# 3. SORT KEY
# REEL: Jam 00-06 ditambah 1 hari (karena di Excel tanggalnya mundur 1 hari dari params)
def create_sort_key_reel(ts):
    if ts.hour < 7:
        return ts + timedelta(days=1)
    return ts

df_reel['Sort_Key'] = df_reel['Timestamp'].apply(create_sort_key_reel)
df_params['Sort_Key'] = df_params['Timestamp']  # Params tidak perlu adjustment

# 4. FILTER BERDASARKAN SORT_KEY RANGE PARAMS
params_sort_min = df_params['Sort_Key'].min()
params_sort_max = df_params['Sort_Key'].max()

df_reel_filtered = df_reel[
    (df_reel['Sort_Key'] >= params_sort_min) & 
    (df_reel['Sort_Key'] <= params_sort_max)
].copy()

df_reel_filtered = df_reel_filtered.sort_values('Sort_Key').reset_index(drop=True)

# 5. DAFTAR VARIABEL
cols_to_avg = [
    'Creping', 'Yankee Speed', 'Pope Reel Speed', 'Yankee Pressure',
    'Stock Flow', 'Stock Consistency', 'Flow Coating', 'Flow Release',
    'Jet Wire Ratio', 'Load KWH Tickling Refiner', 'Coating/(Area.Min)'
]

# 6. LOOPING GROUPBY AVERAGE
results = []

for i in range(len(df_reel_filtered) - 1):
    start_time = df_reel_filtered['Timestamp'].iloc[i]
    end_time = df_reel_filtered['Timestamp'].iloc[i + 1]
    start_sort = df_reel_filtered['Sort_Key'].iloc[i]
    end_sort = df_reel_filtered['Sort_Key'].iloc[i + 1]
    
    mask = (df_params['Sort_Key'] >= start_sort) & (df_params['Sort_Key'] < end_sort)
    df_filtered = df_params.loc[mask]
    
    if len(df_filtered) == 0:
        continue
    
    row = {
        'Start_Time': start_time,
        'End_Time': end_time,
        'Data_Count': len(df_filtered)
    }
    
    for col in cols_to_avg:
        mean_val = df_filtered[col].mean()
        row[f'Mean_{col}'] = round(mean_val, 6) if pd.notna(mean_val) else None
    
    results.append(row)

# 7. HASIL
df_result = pd.DataFrame(results)

# 8. SIMPAN
# df_result.to_excel('grouby_params.xlsx', index=False)
print("\n✅ File disimpan: grouby_params.xlsx")


✅ File disimpan: grouby_params.xlsx


# Join Table

## Joining Df_Results and Reel Data

In [22]:
df_groupby = df_result.copy()
df_groupby.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,Mean_Flow Release,Mean_Jet Wire Ratio,Mean_Load KWH Tickling Refiner,Mean_Coating/(Area.Min)
0,2026-03-03 23:25:00,2026-03-03 00:35:00,35,23.002766,1319.930029,1016.309600,7.999800,1171.583143,3.100743,29.274343,48.175886,0.935000,295.179400,0.000436
1,2026-03-03 00:35:00,2026-03-03 01:55:00,80,21.906992,1319.943275,1030.783375,7.999388,1178.444437,3.101350,29.274275,48.176200,0.932113,293.352262,0.000436
2,2026-03-03 01:55:00,2026-03-03 03:05:00,70,21.594554,1320.486771,1035.328271,7.999129,1183.419886,3.101757,29.286714,48.196586,0.931000,294.043071,0.000436
3,2026-03-03 03:05:00,2026-03-03 04:02:00,57,22.283925,1329.967351,1033.598456,7.998789,1181.181316,3.099526,29.496140,48.541281,0.931000,301.218035,0.000436
4,2026-03-03 04:02:00,2026-03-03 05:35:00,93,22.075381,1339.224710,1043.584946,7.998946,1192.033108,3.103183,29.702301,48.880462,0.932097,295.972054,0.000436


In [23]:
df_reel = reel_pm15.copy()
df_reel.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,2026-03-01,07:03:00,1,2026-03-01 07:03:00,T 14.2.Recycle,98,14.18,0.89,595,403,83,0.139496,27,81.3,Acc Sotiss
1,2026-03-01,08:32:00,1,2026-03-01 08:32:00,T 14.2.Recycle,99,14.78,0.91,788,489,105,0.133249,27,81.3,Acc Sotiss
2,2026-03-01,09:04:00,1,2026-03-01 09:04:00,T 14.2.Recycle,1,14.60,0.94,874,427,106,0.121281,30,81.0,Pending
3,2026-03-01,10:18:00,1,2026-03-01 10:18:00,T 14.2.Recycle,2,14.84,0.90,865,394,115,0.132948,31,81.8,Pending
4,2026-03-01,13:17:00,1,2026-03-01 13:17:00,T 14.2.Recycle,3,15.80,1.03,523,318,75,0.143403,33,81.7,Pending


In [24]:
# Create Join Key in df_groupby
df_groupby['Join_Key_Timestamp'] = df_groupby['End_Time'] #End_Time
# Create Join Key in df_reel
df_reel['Join_Key_Timestamp'] = df_reel['Timestamp']

In [25]:
# Join df_groupby with df_reel on Join_Key_Timestamp
df_joined = pd.merge(df_groupby, df_reel, left_on='Join_Key_Timestamp', right_on='Join_Key_Timestamp', how='inner')
df_joined.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,...,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,2026-03-03 23:25:00,2026-03-03 00:35:00,35,23.002766,1319.930029,1016.309600,7.999800,1171.583143,3.100743,29.274343,...,52,15.00,0.93,814,399,107,0.131450,30,81.3,Acc Sotiss
1,2026-03-03 00:35:00,2026-03-03 01:55:00,80,21.906992,1319.943275,1030.783375,7.999388,1178.444437,3.101350,29.274275,...,53,14.96,0.93,885,442,110,0.124294,28,81.3,Acc Sotiss
2,2026-03-03 01:55:00,2026-03-03 03:05:00,70,21.594554,1320.486771,1035.328271,7.999129,1183.419886,3.101757,29.286714,...,54,15.12,0.93,884,420,115,0.130090,29,81.3,Acc Sotiss
3,2026-03-03 03:05:00,2026-03-03 04:02:00,57,22.283925,1329.967351,1033.598456,7.998789,1181.181316,3.099526,29.496140,...,55,15.10,0.93,796,408,104,0.130653,28,81.3,Acc Sotiss
4,2026-03-03 04:02:00,2026-03-03 05:35:00,93,22.075381,1339.224710,1043.584946,7.998946,1192.033108,3.103183,29.702301,...,56,15.24,0.92,824,458,116,0.140777,29,81.1,Acc Sotiss


## Joining df_joined with BB Table

### Data BB

In [26]:
# Pipeline for Data Reel
def merge_BB_data(file_list):
    dataframes = []
    
    for file in file_list:
        if os.path.exists(file):
            df = pd.read_excel(file, engine='openpyxl')
            dataframes.append(df)
            df.drop(columns=['Grade'], inplace=True)
            print(f"Berhasil memuat: {file} | Shape: {df.shape}")
        else:
            print(f"GAGAL MEMUAT: {file} tidak ditemukan di direktori.")
            
    if not dataframes:
        raise ValueError("Pipeline dihentikan. Tidak ada satupun file yang valid untuk digabungkan.")
        
    master_reel = pd.concat(dataframes, ignore_index=True)
    
    return master_reel

In [27]:
# Eksekusi Pipeline
file_sources = [
    "E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM15.xlsx",
    "E:\Kuliah\Sun Paper Source\Efficiency\BB_April_PM15.xlsx"
]
df_BB = merge_BB_data(file_sources)
df_BB.tail()

Berhasil memuat: E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM15.xlsx | Shape: (31, 10)
Berhasil memuat: E:\Kuliah\Sun Paper Source\Efficiency\BB_April_PM15.xlsx | Shape: (31, 10)


<>:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:4: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:4: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_14800\2911332480.py:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
  "E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM15.xlsx",
C:\Users\user\AppData\Local\Temp\ipykernel_14800\2911332480.py:4: SyntaxWarning: "\K" is an invalid escape sequen

,Date,GSM,Total NBKP,Total LBKP,Total BB Recycle,Sub Total Pulp+Broke,% NBKP,% LBKP,% BB Recycle,% Pulp+Broke
57,2026-04-27,13,0.0,0,48061.139124,48061.139124,0.0,0,100.0,1.311362e+06
58,2026-04-28,"13/13,5",0.0,0,33840.076279,33840.076279,0.0,0,100.0,1.345202e+06
59,2026-04-29,"13,5/16",0.0,0,20471.664924,20471.664924,0.0,0,100.0,1.365673e+06
60,2026-04-30,16,0.0,0,42461.387057,42461.387057,0.0,0,100.0,1.408135e+06
61,2026-05-01,0,0.0,0,0.000000,0.000000,0.0,0,0.0,1.408135e+06


df_BB = pd.read_excel("../Efficiency/BB_Maret_PM15.xlsx", engine='openpyxl')
df_BB.drop(columns=['Grade'], inplace=True)
df_BB.head()

### Last Join

In [28]:
# Standarisasi kolom "date" ke bentuk datetime
df_joined['Date'] = pd.to_datetime(df_joined['Date'])
df_BB['Date'] = pd.to_datetime(df_BB['Date'])

In [29]:
# Joining
df_final = pd.merge(df_joined, df_BB, on='Date', how='inner') #inner
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 1073 entries, 0 to 1072
Data columns (total 39 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Start_Time                      1073 non-null   datetime64[us]
 1   End_Time                        1073 non-null   datetime64[us]
 2   Data_Count                      1073 non-null   int64         
 3   Mean_Creping                    1073 non-null   float64       
 4   Mean_Yankee Speed               1073 non-null   float64       
 5   Mean_Pope Reel Speed            1073 non-null   float64       
 6   Mean_Yankee Pressure            1073 non-null   float64       
 7   Mean_Stock Flow                 1073 non-null   float64       
 8   Mean_Stock Consistency          1073 non-null   float64       
 9   Mean_Flow Coating               1073 non-null   float64       
 10  Mean_Flow Release               1073 non-null   float64       
 11  Mean_Jet Wire R

### Convert Final Data to Excel

#### Group by Grade

In [30]:
# Membaca file Excel
df = df_final

# Memisahkan data berdasarkan awalan pada kolom Grade
df_toilet = df[df['Grade'].str.startswith('T', na=False) & 
               ~df['Grade'].str.startswith('TW', na=False)]
df_towel = df[df['Grade'].str.startswith('TW', na=False)]
df_facial = df[df['Grade'].str.startswith('FC', na=False)]

# Menampilkan jumlah data
print("Jumlah data Toilet :", len(df_toilet))
print("Jumlah data Towel  :", len(df_towel))
print("Jumlah data Facial :", len(df_facial))



Jumlah data Toilet : 805
Jumlah data Towel  : 185
Jumlah data Facial : 83


Simpan ke file Excel terpisah

df_toilet.to_excel("Final_PM15-Toilet.xlsx", index=False)
df_towel.to_excel("Final_PM15-Towel.xlsx", index=False)
df_facial.to_excel("Final_PM15-Facial.xlsx", index=False)

df_final.to_excel('Final_PM15.xlsx', index=False)